In [ ]:
-- 실행 컨텍스트 설정
USE ROLE ACCOUNTADMIN;
USE DATABASE DEMO;
USE SCHEMA ML_DEMO;

# Module 6: ML Jobs & Task Graph 파이프라인 오케스트레이션

## 학습 목표

이 모듈에서는 Snowflake의 **ML Jobs**와 **Task Graph**를 활용하여 ML 파이프라인을 프로덕션 환경에 배포하고 자동화하는 방법을 학습합니다.

| 단계 | 도구 | 목적 |
|------|------|------|
| 개발 (Dev) | Snowflake Notebook | 탐색, 실험, 프로토타이핑 |
| 실행 (Prod Compute) | ML Jobs (`@remote`) | 격리된 컨테이너에서 대규모 학습 |
| 자동화 (Automation) | Task Graph (DAG) | 스케줄 기반 파이프라인 오케스트레이션 |

## ML 라이프사이클 아키텍처

```
┌─────────────────────────────────────────────────────────────────┐
│                     ML Production Pipeline                      │
├──────────────┬──────────────────────────────┬───────────────────┤
│  Notebook    │        ML Jobs               │   Task Graph      │
│  (Dev/탐색)  │  (@remote 데코레이터)         │   (DAG 자동화)    │
│              │                              │                   │
│  • 데이터    │  • Compute Pool에서 실행      │  • 일별/주별 스케줄│
│    탐색      │  • 격리된 컨테이너 환경       │  • 의존성 관리    │
│  • 모델      │  • 비동기 실행               │  • 재시도 정책    │
│    실험      │  • 로그 수집                 │  • 알림 설정      │
│  • 검증      │  • 상태 모니터링             │  • 시각화 (UI)    │
└──────────────┴──────────────────────────────┴───────────────────┘
```

---
**데모 시나리오**: TPC-H 데이터 기반 HIGH-VALUE 고객 LTV 회귀 모델의 전체 프로덕션 파이프라인 구축

## 1. Setup & 라이브러리 임포트

In [ ]:
# 핵심 라이브러리 임포트
import json
import time
from datetime import timedelta

# Snowpark
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F

# ML Jobs
from snowflake.ml.jobs import remote

# Task Graph (DAG)
from snowflake.core import Root
from snowflake.core.task import Task
from snowflake.core.task.dagv1 import DAGTask, DAG

# 세션 초기화
session = get_active_session()

# 환경 설정
DATABASE     = "DEMO"
SCHEMA       = "ML_DEMO"
WAREHOUSE    = "COMPUTE_WH"
COMPUTE_POOL = "SYSTEM_COMPUTE_POOL_CPU"
STAGE_NAME   = "@DEMO.ML_DEMO.ML_STAGE"
MODEL_NAME   = "CUSTOMER_LTV_PREDICTOR"

print(f"세션 연결: {session.get_current_account()}")
print(f"데이터베이스: {DATABASE}.{SCHEMA}")
print(f"컴퓨팅 풀: {COMPUTE_POOL}")

---
## 2. Stage 생성

### ML Stage의 역할

ML Jobs는 코드와 의존성 파일을 **Snowflake Internal Stage**에 업로드하여 Compute Pool 컨테이너에서 실행합니다.

```
Local Code / Notebook
        │
        ▼  (직렬화 & 업로드)
@DEMO.ML_DEMO.ML_STAGE
        │
        ▼  (다운로드 & 실행)
Compute Pool Container (SYSTEM_COMPUTE_POOL_CPU)
        │
        ▼  (결과 저장)
Snowflake Model Registry
```

- **코드 스테이징**: `@remote` 데코레이터가 함수 코드를 자동으로 Stage에 업로드
- **의존성 패키징**: 필요한 패키지 정보도 함께 전송
- **결과 공유**: 모델, 로그, 아티팩트를 Stage를 통해 반환

In [ ]:
# ML Stage 생성 (없는 경우 생성)
session.sql("""
    CREATE STAGE IF NOT EXISTS DEMO.ML_DEMO.ML_STAGE
        COMMENT = 'ML Jobs 코드 스테이징 및 아티팩트 저장용 Stage'
""").collect()

print("ML_STAGE 생성 완료")

# Stage 정보 확인
stage_info = session.sql("SHOW STAGES IN SCHEMA DEMO.ML_DEMO").collect()
for row in stage_info:
    if "ML_STAGE" in row["name"]:
        print(f"Stage: {row['name']}, URL: {row['url']}")

---
## 3. ML Jobs: `@remote` 데코레이터

### `@remote` 데코레이터란?

`@remote`는 Python 함수를 **Snowflake Compute Pool**에서 비동기적으로 실행하는 데코레이터입니다.

| 항목 | Notebook 실행 | ML Jobs (`@remote`) |
|------|--------------|---------------------|
| 실행 환경 | Notebook 커널 | Compute Pool 컨테이너 |
| 리소스 격리 | 공유 | 전용 할당 |
| 비동기 실행 | ✗ (블로킹) | ✓ (Job 객체 반환) |
| 대규모 처리 | 제한적 | 적합 (GPU 포함) |
| 로그 수집 | 즉시 출력 | `job.get_logs()` |

### 파라미터 설명
```python
@remote(
    session=session,                           # 활성 Snowpark 세션
    compute_pool="SYSTEM_COMPUTE_POOL_CPU",    # 실행할 Compute Pool 이름
    stage_name="@DEMO.ML_DEMO.ML_STAGE",       # 코드 업로드용 Stage
)
```

In [ ]:
@remote(
    session=session,
    compute_pool=COMPUTE_POOL,
    stage_name=STAGE_NAME,
)
def train_high_value_model(config: dict):
    """
    HIGH-VALUE 고객 LTV 회귀 모델 학습 함수.
    이 함수는 Compute Pool 컨테이너에서 실행됩니다.
    """
    # --- 컨테이너 내부에서 실행되는 코드 ---
    from snowflake.snowpark.context import get_active_session
    from snowflake.ml.modeling.xgboost import XGBRegressor
    from snowflake.ml.modeling.pipeline import Pipeline
    from snowflake.ml.modeling.preprocessing import StandardScaler, OrdinalEncoder
    from snowflake.ml.registry import Registry
    import snowflake.snowpark.functions as F

    session = get_active_session()

    print("[Job] 학습 데이터 로딩...")
    # Module 4 패턴: CUSTOMER_FEATURES (raw) 로드 — OrdinalEncoder+StandardScaler를
    # 파이프라인에 포함시켜 학습 데이터에만 fit, 데이터 누수 방지
    train_df = session.table("DEMO.ML_DEMO.CUSTOMER_FEATURES")

    CAT_COLS = ["C_MKTSEGMENT"]
    NUM_COLS = [
        "C_ACCTBAL", "TOTAL_ORDERS", "AVG_ORDER_VALUE",
        "TOTAL_REVENUE", "AVG_DISCOUNT", "AVG_QUANTITY",
        "DAYS_SINCE_LAST_ORDER", "C_NATIONKEY",
    ]
    FEATURE_COLS = CAT_COLS + NUM_COLS
    LABEL_COL = "FUTURE_12M_REVENUE"

    # 하이퍼파라미터 설정 (config에서 읽기)
    max_depth    = config.get("max_depth", 6)
    n_estimators = config.get("n_estimators", 100)
    learning_rate = config.get("learning_rate", 0.1)

    print(f"[Job] 하이퍼파라미터: max_depth={max_depth}, n_estimators={n_estimators}, lr={learning_rate}")

    # 파이프라인 구성 (OrdinalEncoder → StandardScaler → XGBRegressor)
    # handle_unknown='use_encoded_value', unknown_value=-1 필수 (미관측 범주 런타임 오류 방지)
    pipeline = Pipeline(steps=[
        ("encoder", OrdinalEncoder(
            input_cols=CAT_COLS,
            output_cols=CAT_COLS,
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        )),
        ("scaler", StandardScaler(input_cols=NUM_COLS, output_cols=NUM_COLS)),
        ("regressor", XGBRegressor(
            input_cols=FEATURE_COLS,
            label_cols=[LABEL_COL],
            output_cols=["PREDICTED_LTV"],
            max_depth=max_depth,
            n_estimators=n_estimators,
            learning_rate=learning_rate,
        )),
    ])

    print("[Job] 모델 학습 시작...")
    pipeline.fit(train_df)
    print("[Job] 모델 학습 완료")

    # 모델 레지스트리에 등록
    # sample_input_data: Snowpark DataFrame 필수 (pandas 불가), .limit(100)으로 경량화
    registry = Registry(session=session, database_name="DEMO", schema_name="ML_DEMO")
    model_version = registry.log_model(
        model=pipeline,
        model_name="CUSTOMER_LTV_PREDICTOR",
        version_name=config.get("version_name", "V_LATEST"),
        comment=f"ML Job으로 학습된 모델 | max_depth={max_depth}, n_estimators={n_estimators}",
        sample_input_data=train_df.select(FEATURE_COLS).limit(100),
    )
    print(f"[Job] 모델 등록 완료: {model_version.model_name} @ {model_version.version_name}")
    return {"status": "success", "model": model_version.model_name, "version": model_version.version_name}

### Job 실행

`@remote` 데코레이터가 적용된 함수를 호출하면 즉시 **Job 객체**를 반환하고, 실제 실행은 Compute Pool에서 비동기로 진행됩니다.

In [ ]:
# Job 제출 (비동기 실행)
train_config = {
    "max_depth": 6,
    "n_estimators": 100,
    "learning_rate": 0.1,
    "version_name": "V_JOB_001",
}

print("ML Job 제출 중...")
job = train_high_value_model(train_config)

print(f"Job ID   : {job.id}")
print(f"초기 상태: {job.status}")
print()
print("Job이 Compute Pool에서 비동기로 실행 중입니다.")
print("job.wait()로 완료를 기다리거나, job.status로 폴링할 수 있습니다.")

---
## 4. Job 상태 모니터링

### Job 상태 값

| 상태 | 의미 |
|------|------|
| `PENDING` | 큐에서 대기 중 (리소스 할당 전) |
| `RUNNING` | Compute Pool에서 실행 중 |
| `DONE` | 성공적으로 완료 |
| `FAILED` | 오류로 실패 |
| `CANCELLED` | 사용자가 취소 |

### Snowsight에서 Job 확인

1. Snowsight 좌측 메뉴 → **Data Engineering** → **ML Jobs**
2. Job 목록에서 실행 중인 Job 클릭
3. 로그, 상태, 소요 시간 실시간 확인

In [ ]:
# 폴링 방식으로 상태 모니터링
print("Job 상태 모니터링 (10초 간격 폴링)")
print("-" * 40)

poll_count = 0
while True:
    status = job.status
    poll_count += 1
    print(f"[{poll_count:02d}] 상태: {status}")

    if status in ("DONE", "FAILED", "CANCELLED"):
        break

    time.sleep(10)

print()
print(f"최종 상태: {status}")

In [ ]:
# 로그 확인
print("=== Job 실행 로그 ===")
logs = job.get_logs()
print(logs)

In [ ]:
# 간단한 완료 대기 (블로킹 방식)
# job.wait()는 Job이 완료될 때까지 대기하고 최종 상태를 반환합니다

# 아래는 이미 완료된 Job에서 wait()를 호출하는 예시입니다
# final_status = job.wait()
# print(f"최종 상태: {final_status}")

# 완료 후 결과 확인
if job.status == "DONE":
    print("Job 성공적으로 완료!")
    print()
    # 레지스트리에 등록된 모델 버전 확인
    from snowflake.ml.registry import Registry
    registry = Registry(session=session, database_name="DEMO", schema_name="ML_DEMO")
    model_ref = registry.get_model(MODEL_NAME)
    print("등록된 모델 버전 목록:")
    for v in model_ref.versions():
        print(f"  {v.version_name}")
elif job.status == "FAILED":
    print("Job 실패. 로그를 확인하세요:")
    print(job.get_logs())

---
## 5. Task Graph (DAG) 구성

### Task Graph란?

**Task Graph**는 Snowflake의 Task를 DAG(Directed Acyclic Graph) 형태로 연결하여 복잡한 파이프라인을 구성하고 스케줄링하는 기능입니다.

### ML 파이프라인 DAG 설계

```
┌─────────────────────────────────────────────────────────────┐
│              ML_TRAINING_DAG  (매일 새벽 2시 실행)           │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   [1] data_prep          CALL PREPARE_FEATURES()           │
│          │                                                  │
│          ▼                                                  │
│   [2] model_train        CALL TRAIN_MODEL_PROC()           │
│          │                                                  │
│          ▼                                                  │
│   [3] batch_score        CALL SCORE_CUSTOMERS()            │
│          │                                                  │
│          ▼                                                  │
│   [4] monitor_refresh    CALL REFRESH_MONITOR()            │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 각 Task의 역할

| Task | 역할 |
|------|------|
| `data_prep` | 원시 TPC-H 데이터에서 피처 생성 및 학습/평가 분리 |
| `model_train` | XGBoost 모델 학습 및 Model Registry 등록 |
| `batch_score` | 전체 고객 데이터에 최신 모델 적용, 예측 저장 |
| `monitor_refresh` | 모델 성능 지표 갱신, 드리프트 감지 대시보드 업데이트 |

In [ ]:
# 보조 저장 프로시저 생성 (DAG Task에서 호출)

# 1. 피처 준비 프로시저
# Module 2 패턴: LINEITEM은 L_ORDERKEY 레벨에서 먼저 집계 후 고객 레벨로 JOIN
# → 직접 고객 레벨 집계 시 LINEITEM 팬아웃으로 AVG_DISCOUNT/AVG_QUANTITY 왜곡 방지
session.sql("""
    CREATE OR REPLACE PROCEDURE DEMO.ML_DEMO.PREPARE_FEATURES()
    RETURNS VARCHAR
    LANGUAGE SQL
    AS
    $$
    BEGIN
        -- Step 1: LINEITEM을 주문 레벨로 먼저 집계 (팬아웃 방지)
        CREATE OR REPLACE TEMPORARY TABLE TMP_ORDER_AGG AS
        SELECT
            L_ORDERKEY,
            SUM(L_EXTENDEDPRICE * (1 - L_DISCOUNT))  AS ORDER_REVENUE,
            AVG(L_DISCOUNT)                           AS ORDER_AVG_DISCOUNT,
            AVG(L_QUANTITY)                           AS ORDER_AVG_QUANTITY
        FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.LINEITEM
        GROUP BY L_ORDERKEY;

        -- Step 2: 고객 레벨 피처 생성 (Module 02/04와 동일 스키마)
        CREATE OR REPLACE TABLE DEMO.ML_DEMO.CUSTOMER_FEATURES AS
        SELECT
            c.C_CUSTKEY,
            c.C_MKTSEGMENT,
            c.C_ACCTBAL,
            c.C_NATIONKEY,
            COUNT(DISTINCT o.O_ORDERKEY)                          AS TOTAL_ORDERS,
            AVG(oa.ORDER_REVENUE)                                 AS AVG_ORDER_VALUE,
            SUM(oa.ORDER_REVENUE)                                 AS TOTAL_REVENUE,
            AVG(oa.ORDER_AVG_DISCOUNT)                            AS AVG_DISCOUNT,
            AVG(oa.ORDER_AVG_QUANTITY)                             AS AVG_QUANTITY,
            DATEDIFF('day', MAX(o.O_ORDERDATE), CURRENT_DATE())   AS DAYS_SINCE_LAST_ORDER,
            -- FUTURE_12M_REVENUE: TOTAL_REVENUE 중앙값 기준 이진 레이블
            CASE
                WHEN SUM(oa.ORDER_REVENUE) >
                     MEDIAN(SUM(oa.ORDER_REVENUE)) OVER () THEN 1
                ELSE 0
            END                                                   AS FUTURE_12M_REVENUE
        FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER  c
        JOIN SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS    o  ON c.C_CUSTKEY  = o.O_CUSTKEY
        JOIN TMP_ORDER_AGG                             oa ON o.O_ORDERKEY = oa.L_ORDERKEY
        GROUP BY c.C_CUSTKEY, c.C_MKTSEGMENT, c.C_ACCTBAL, c.C_NATIONKEY
        HAVING TOTAL_ORDERS > 0;

        RETURN 'PREPARE_FEATURES 완료: ' || (SELECT COUNT(*) FROM DEMO.ML_DEMO.CUSTOMER_FEATURES) || ' rows';
    END;
    $$
""").collect()

# 2. 배치 스코어링 프로시저
session.sql("""
    CREATE OR REPLACE PROCEDURE DEMO.ML_DEMO.SCORE_CUSTOMERS()
    RETURNS VARCHAR
    LANGUAGE SQL
    AS
    $$
    BEGIN
        -- 최신 모델로 전체 고객 스코어링
        CREATE OR REPLACE TABLE DEMO.ML_DEMO.CUSTOMER_PREDICTED_LTVS AS
        SELECT
            C_CUSTKEY,
            DEMO.ML_DEMO.CUSTOMER_LTV_PREDICTOR!PREDICT(
                C_MKTSEGMENT,
                C_ACCTBAL, TOTAL_ORDERS, AVG_ORDER_VALUE,
                TOTAL_REVENUE, AVG_DISCOUNT, AVG_QUANTITY,
                DAYS_SINCE_LAST_ORDER, C_NATIONKEY
            ):PREDICTED_LTV::INT AS PREDICTED_LTV,
            CURRENT_TIMESTAMP() AS SCORED_AT
        FROM DEMO.ML_DEMO.CUSTOMER_FEATURES;

        RETURN 'SCORE_CUSTOMERS 완료: ' || (SELECT COUNT(*) FROM DEMO.ML_DEMO.CUSTOMER_PREDICTED_LTVS) || ' rows scored';
    END;
    $$
""").collect()

# 3. 모니터 갱신 프로시저
# ModelMonitor import는 런타임에 따라 가용성이 다르므로 try/except 필수
session.sql("""
    CREATE OR REPLACE PROCEDURE DEMO.ML_DEMO.REFRESH_MONITOR()
    RETURNS VARCHAR
    LANGUAGE PYTHON
    RUNTIME_VERSION = '3.11'
    PACKAGES = ('snowflake-snowpark-python', 'snowflake-ml-python')
    HANDLER = 'refresh'
    AS
    $$
def refresh(session):
    try:
        from snowflake.ml.monitoring import ModelMonitor
        monitor = ModelMonitor(
            session=session,
            name='HIGH_VALUE_MONITOR',
            database_name='DEMO',
            schema_name='ML_DEMO',
        )
        monitor.refresh()
        return 'REFRESH_MONITOR 완료 (ModelMonitor): ' + str(__import__('datetime').datetime.utcnow())
    except (ImportError, AttributeError, Exception) as e:
        # ModelMonitor 미사용 환경 또는 모니터 미생성 시 fallback
        return 'REFRESH_MONITOR fallback (모니터 없음): ' + str(e)
    $$
""").collect()

print("보조 저장 프로시저 3개 생성 완료:")
print("  - DEMO.ML_DEMO.PREPARE_FEATURES()")
print("  - DEMO.ML_DEMO.SCORE_CUSTOMERS()")
print("  - DEMO.ML_DEMO.REFRESH_MONITOR()")

In [ ]:
# 모델 학습 저장 프로시저 (Task에서 직접 호출 가능하도록)
session.sql("""
    CREATE OR REPLACE PROCEDURE DEMO.ML_DEMO.TRAIN_MODEL_PROC()
    RETURNS VARCHAR
    LANGUAGE PYTHON
    RUNTIME_VERSION = '3.11'
    PACKAGES = ('snowflake-snowpark-python', 'snowflake-ml-python')
    HANDLER = 'train'
    AS
    $$
def train(session):
    from snowflake.ml.modeling.xgboost import XGBRegressor
    from snowflake.ml.modeling.pipeline import Pipeline
    from snowflake.ml.modeling.preprocessing import StandardScaler, OrdinalEncoder
    from snowflake.ml.registry import Registry

    CAT_COLS = ["C_MKTSEGMENT"]
    NUM_COLS = [
        "C_ACCTBAL", "TOTAL_ORDERS", "AVG_ORDER_VALUE",
        "TOTAL_REVENUE", "AVG_DISCOUNT", "AVG_QUANTITY",
        "DAYS_SINCE_LAST_ORDER", "C_NATIONKEY",
    ]
    FEATURE_COLS = CAT_COLS + NUM_COLS
    LABEL_COL = "FUTURE_12M_REVENUE"

    # Module 4 패턴: CUSTOMER_FEATURES (raw) 로드
    train_df = session.table("DEMO.ML_DEMO.CUSTOMER_FEATURES")

    pipeline = Pipeline(steps=[
        ("encoder", OrdinalEncoder(
            input_cols=CAT_COLS,
            output_cols=CAT_COLS,
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        )),
        ("scaler", StandardScaler(input_cols=NUM_COLS, output_cols=NUM_COLS)),
        ("regressor", XGBRegressor(
            input_cols=FEATURE_COLS,
            label_cols=[LABEL_COL],
            output_cols=["PREDICTED_LTV"],
            max_depth=6,
            n_estimators=100,
        )),
    ])

    pipeline.fit(train_df)

    registry = Registry(session=session, database_name="DEMO", schema_name="ML_DEMO")
    registry.log_model(
        model=pipeline,
        model_name="CUSTOMER_LTV_PREDICTOR",
        version_name="V_TASK",
        comment="Task Graph에서 자동 학습된 모델",
        sample_input_data=train_df.select(FEATURE_COLS).limit(100),
    )
    return "TRAIN_MODEL_PROC 완료"
    $$
""").collect()

print("TRAIN_MODEL_PROC 프로시저 생성 완료")



> **프로덕션 권장 패턴**
> 
> 이 데모에서는 학습 단계를 **Warehouse 기반 Stored Procedure**로 구현합니다 (교육 단순화 목적).
> 실제 프로덕션에서는 대규모 데이터·GPU 학습이 필요하므로 아래 패턴을 권장합니다:
> 
> | 방법 | 설명 |
> |------|------|
> | `submit_file()` | 학습 스크립트를 Stage에 업로드 후 Compute Pool에서 실행 |
> | `@remote` + SP wrapper | SP 내부에서 `@remote` 함수를 호출하여 Compute Pool 실행 |

**방법 1: `submit_file()` 패턴**

```python
# SP 내부에서 Stage의 학습 스크립트를 Compute Pool에 제출
def train(session):
    from snowflake.ml.jobs import submit_file
    job = submit_file("@ML_STAGE/train.py", "MY_GPU_POOL",
                      stage_name="@ML_STAGE", session=session)
    job.wait()  # 완료 대기
    return f"완료: {job.status}"
```

**방법 2: `@remote` + SP wrapper 패턴**

```python
# SP가 @remote 데코레이터 함수를 호출 → 자동으로 코드 직렬화 & Compute Pool 실행
def train(session):
    from snowflake.ml.jobs import remote

    @remote(
        session=session,
        compute_pool="MY_GPU_POOL",
        stage_name="@ML_STAGE",
    )
    def _train_on_compute_pool():
        # 이 함수 전체가 Compute Pool 컨테이너에서 실행됨
        from snowflake.snowpark.context import get_active_session
        from snowflake.ml.modeling.xgboost import XGBRegressor
        ...
        pipeline.fit(train_df)
        registry.log_model(pipeline, ...)
        return "학습 완료"

    job = _train_on_compute_pool()  # Job 제출 (비동기)
    job.wait()                      # 완료 대기
    return f"완료: {job.status}"
```

> 두 방법 모두 Task Graph의 `CALL TRAIN_MODEL_PROC()` 형태로 스케줄링 가능합니다.


### DAG 구성 (Python API)

`snowflake.core` Python API를 사용하면 코드로 DAG를 정의하고 배포할 수 있습니다.

In [ ]:
# Root 객체 초기화 (Snowflake Core API)
root = Root(session)

# 스키마 리소스 참조
schema_ref = root.databases[DATABASE].schemas[SCHEMA]

# DAG 정의
# - schedule: timedelta(days=1) → 매일 실행
# - warehouse: DAG 루트 Task에 사용할 Warehouse
# - stage_location: Task 코드 저장 Stage

train_model_sql = "CALL DEMO.ML_DEMO.TRAIN_MODEL_PROC()"

with DAG(
    "ML_TRAINING_DAG",
    schedule=timedelta(days=1),
    warehouse=WAREHOUSE,
    stage_location=STAGE_NAME,
    comment="HIGH-VALUE 고객 LTV 회귀 모델 일별 재학습 파이프라인",
) as dag:

    # Task 정의 (각 Step에 사용할 SQL 저장 프로시저 지정)
    task_prep = DAGTask(
        name="data_prep",
        definition="CALL DEMO.ML_DEMO.PREPARE_FEATURES()",
        warehouse=WAREHOUSE,
        comment="원시 데이터에서 ML 피처 생성",
    )
    task_train = DAGTask(
        name="model_train",
        definition=train_model_sql,
        warehouse=WAREHOUSE,
        comment="XGBoost 모델 재학습 및 레지스트리 등록",
    )
    task_score = DAGTask(
        name="batch_score",
        definition="CALL DEMO.ML_DEMO.SCORE_CUSTOMERS()",
        warehouse=WAREHOUSE,
        comment="전체 고객 배치 스코어링",
    )
    task_monitor = DAGTask(
        name="monitor_refresh",
        definition="CALL DEMO.ML_DEMO.REFRESH_MONITOR()",
        warehouse=WAREHOUSE,
        comment="모델 모니터 지표 갱신",
    )

    # 의존성 체인 정의 (>> 연산자: 선행 → 후행)
    task_prep >> task_train >> task_score >> task_monitor

print("DAG 정의 완료: ML_TRAINING_DAG")
print(f"스케줄   : 매일 (timedelta(days=1))")
print(f"Task 수  : 4개")
print(f"실행 순서: data_prep → model_train → batch_score → monitor_refresh")

---
## 6. DAG 배포 및 실행

### 배포 vs 실행

| 작업 | 설명 |
|------|------|
| **배포** (Deploy) | DAG 정의를 Snowflake에 Task 객체로 등록 |
| **실행** (Execute) | 스케줄과 무관하게 즉시 수동 실행 트리거 |
| **활성화** (Resume) | 자동 스케줄 실행 활성화 |
| **중단** (Suspend) | 자동 스케줄 실행 중단 |

### Snowsight에서 DAG 시각화

1. Snowsight → **Data Engineering** → **Task Graphs**
2. `ML_TRAINING_DAG` 클릭
3. **Graph** 탭: Task 의존성 시각화 확인
4. **Run History** 탭: 과거 실행 이력, 각 Task 소요 시간 확인

In [ ]:
# DAG 배포
# DAGOperation을 사용하여 DAG를 Snowflake에 Task 객체로 등록합니다
from snowflake.core.task.dagv1 import DAGOperation
from snowflake.core._common import CreateMode

schema = root.databases[DATABASE].schemas[SCHEMA]
dag_op = DAGOperation(schema)
dag_op.deploy(dag, mode=CreateMode.or_replace)

print("DAG 배포 완료!")
print()

# 배포된 Task 목록 확인
task_list = session.sql("""
    SHOW TASKS IN SCHEMA DEMO.ML_DEMO
""").collect()

print("배포된 Task 목록:")
print(f"{"이름":<30} {"상태":<12} {"스케줄":<20}")
print("-" * 65)
for task in task_list:
    name     = task["name"]
    state    = task["state"]
    schedule = task.as_dict().get("schedule", "-") or "-"
    print(f"{name:<30} {state:<12} {schedule:<20}")

In [ ]:
# DAG 즉시 실행 (수동 트리거)
# 스케줄을 기다리지 않고 지금 바로 실행
print("DAG 수동 실행 중...")

dag_task_ref = schema_ref.tasks["ML_TRAINING_DAG"]
dag_task_ref.execute()

print("DAG 실행 요청 완료. Task가 순차적으로 실행됩니다.")
print()
print("실행 상태는 Snowsight > Data Engineering > Task Graphs에서 확인하세요.")

In [ ]:
# Task Graph 실행 이력 확인
import pandas as pd

dag_runs = session.sql("""
    SELECT
        NAME,
        STATE,
        SCHEDULED_TIME,
        COMPLETED_TIME,
        DATEDIFF('second', SCHEDULED_TIME, COMPLETED_TIME) AS DURATION_SECONDS,
        ERROR_MESSAGE
    FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
        SCHEDULED_TIME_RANGE_START => DATEADD('hour', -24, CURRENT_TIMESTAMP()),
        TASK_NAME => 'ML_TRAINING_DAG'
    ))
    ORDER BY SCHEDULED_TIME DESC
    LIMIT 10
""").to_pandas()

if dag_runs.empty:
    print("아직 실행 이력이 없습니다. DAG 실행 후 다시 확인하세요.")
else:
    print("최근 DAG 실행 이력:")
    print(dag_runs.to_string(index=False))

In [ ]:
# 모든 Task의 실행 이력 확인 (개별 Task 레벨)
all_task_history = session.sql("""
    SELECT
        NAME,
        STATE,
        SCHEDULED_TIME,
        COMPLETED_TIME,
        DATEDIFF('second', SCHEDULED_TIME, COMPLETED_TIME) AS DURATION_SEC
    FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
        SCHEDULED_TIME_RANGE_START => DATEADD('hour', -24, CURRENT_TIMESTAMP())
    ))
    WHERE NAME IN ('ML_TRAINING_DAG', 'DATA_PREP', 'MODEL_TRAIN', 'BATCH_SCORE', 'MONITOR_REFRESH')
    ORDER BY SCHEDULED_TIME DESC, NAME
    LIMIT 20
""").to_pandas()

if all_task_history.empty:
    print("실행 이력이 없습니다.")
else:
    print("개별 Task 실행 이력:")
    print(all_task_history.to_string(index=False))

---
## 7. Scheduled Notebooks

### Task Graph vs Scheduled Notebook

모든 파이프라인이 완전한 Task Graph를 필요로 하지는 않습니다. **Scheduled Notebook**은 더 간단한 정기 실행 요구사항에 적합합니다.

| 항목 | Task Graph | Scheduled Notebook |
|------|-----------|--------------------|
| 설정 복잡도 | 높음 (DAG 코드 필요) | 낮음 (UI 버튼 클릭) |
| 의존성 관리 | 완전한 DAG 지원 | 단일 노트북 실행 |
| 재시도 정책 | Task 레벨 설정 가능 | 없음 |
| 적합한 사용 | 복잡한 멀티 스텝 파이프라인 | 단순 보고서, 데이터 갱신 |
| 알림 설정 | 상세 알림 가능 | 기본 알림만 |

### Snowsight에서 노트북 스케줄 설정 방법

1. Snowsight에서 노트북 열기
2. 우측 상단 **⚡ Schedule** 버튼 클릭
3. 스케줄 유형 선택:
   - **Minute**: N분마다
   - **Hour**: 매시간
   - **Day**: 매일 특정 시간
   - **Week**: 매주 특정 요일/시간
   - **CRON**: 사용자 정의 CRON 표현식
4. Warehouse 선택 및 활성화

### 적합한 사용 사례 (이 데모)

- 매일 새벽 고객 세그먼트 요약 보고서 생성
- 주간 모델 성능 지표 검토 노트북
- 월간 데이터 품질 검사

In [ ]:
# Snowflake API로 노트북 스케줄 조회 (현재 계정의 노트북 확인)
scheduled_nb = session.sql("""
    SELECT
        NOTEBOOK_NAME,
        NOTEBOOK_SCHEMA,
        NOTEBOOK_OWNER,
        NOTEBOOK_QUERY_WAREHOUSE,
        LAST_ALTERED
    FROM INFORMATION_SCHEMA.NOTEBOOKS
    WHERE NOTEBOOK_CATALOG = 'DEMO'
      AND NOTEBOOK_SCHEMA = 'ML_DEMO'
""").to_pandas()

if scheduled_nb.empty:
    print("ML_DEMO 스키마에 노트북이 없습니다.")
    print()
    print("Snowsight UI에서 노트북을 열고 'Schedule' 버튼으로 스케줄을 설정할 수 있습니다.")
else:
    print("ML_DEMO 노트북 목록:")
    print(scheduled_nb.to_string(index=False))
    print()
    print("💡 스케줄 설정: Snowsight에서 노트북 열기 → 우측 상단 Schedule 버튼")

---
## 8. CI/CD 통합 개념

### DEV → PROD 스키마 분리 전략

```
DEMO.ML_DEMO_DEV    ← 개발자 실험 환경 (Notebook, 작은 샘플)
DEMO.ML_DEMO_STAGING← CI 테스트 환경 (중간 규모 데이터)
DEMO.ML_DEMO        ← 프로덕션 (전체 TPC-H SF1 데이터)
```

### GitHub Actions DAG 배포 패턴

```yaml
# .github/workflows/deploy_ml_pipeline.yml
name: Deploy ML Pipeline DAG

on:
  push:
    branches: [main]
    paths: ['ml_pipeline/**']

jobs:
  deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Install Snowflake CLI
        run: pip install snowflake-cli-labs

      - name: Deploy DAG to Snowflake
        env:
          SNOWFLAKE_ACCOUNT: ${{ secrets.SNOWFLAKE_ACCOUNT }}
          SNOWFLAKE_USER:    ${{ secrets.SNOWFLAKE_USER }}
          SNOWFLAKE_PASSWORD: ${{ secrets.SNOWFLAKE_PASSWORD }}
        run: |
          python deploy_dag.py \
            --schema ML_DEMO \
            --warehouse COMPUTE_WH
```

### deploy_dag.py 패턴

```python
# deploy_dag.py (CI 서버에서 실행)
import argparse
from snowflake.snowpark import Session
from snowflake.core import Root
from snowflake.core.task.dagv1 import DAGTask, DAG
from datetime import timedelta

def deploy(schema: str, warehouse: str):
    session = Session.builder.configs({
        "account":  os.environ["SNOWFLAKE_ACCOUNT"],
        "user":     os.environ["SNOWFLAKE_USER"],
        "password": os.environ["SNOWFLAKE_PASSWORD"],
        "database": "DEMO",
        "schema":   schema,
    }).create()

    root = Root(session)

    with DAG("ML_TRAINING_DAG", schedule=timedelta(days=1),
             warehouse=warehouse) as dag:
        # ... Task 정의 ...
        pass

    DAGOperation(schema).deploy(dag, mode=CreateMode.or_replace)
    print(f"DAG '{schema}.ML_TRAINING_DAG' 배포 완료")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--schema",    default="ML_DEMO")
    parser.add_argument("--warehouse", default="COMPUTE_WH")
    args = parser.parse_args()
    deploy(args.schema, args.warehouse)
```

### 핵심 원칙

1. **코드 as IaC**: DAG 정의를 버전 관리 (Git)
2. **환경 분리**: DEV → STAGING → PROD 스키마 분리
3. **시크릿 관리**: GitHub Secrets, Snowflake Secrets Object 활용
4. **테스트 자동화**: CI에서 DAG 구문 검사, 소규모 데이터 테스트
5. **모델 버전 태깅**: Git 커밋 해시 → Model Registry 버전 매핑

In [ ]:
# 환경별 설정 관리 예시
ENV_CONFIG = {
    "dev": {
        "schema":    "ML_DEMO_DEV",
        "warehouse": "COMPUTE_WH",
        "schedule":  None,            # 개발 중에는 스케줄 비활성화
        "sample_frac": 0.1,           # 10% 샘플로 빠른 개발
    },
    "staging": {
        "schema":    "ML_DEMO_STAGING",
        "warehouse": "COMPUTE_WH",
        "schedule":  timedelta(days=7),  # 주 1회 CI 검증
        "sample_frac": 0.5,
    },
    "prod": {
        "schema":    "ML_DEMO",
        "warehouse": "COMPUTE_WH",
        "schedule":  timedelta(days=1),  # 매일 실행
        "sample_frac": 1.0,              # 전체 데이터
    },
}

# 현재 환경 표시
print("환경별 파이프라인 설정:")
print()
for env, cfg in ENV_CONFIG.items():
    schedule_str = str(cfg['schedule']) if cfg['schedule'] else "수동 실행만"
    print(f"[{env.upper():8}]  스키마: {cfg['schema']:<25}  스케줄: {schedule_str}")

---
## 정리 & 다음 단계

### Module 6 핵심 요약

| 개념 | 도구 | 주요 API |
|------|------|----------|
| 격리된 ML 학습 실행 | ML Jobs | `@remote`, `job.status`, `job.wait()` |
| 코드 스테이징 | Internal Stage | `CREATE STAGE`, `@DEMO.ML_DEMO.ML_STAGE` |
| 파이프라인 오케스트레이션 | Task Graph (DAG) | `DAG`, `DAGTask`, `>>` 연산자 |
| DAG 배포 | snowflake-core | `DAGOperation(schema).deploy(dag)` |
| 수동 실행 | Task API | `schema.tasks["..."].execute()` |
| 실행 이력 | INFORMATION_SCHEMA | `TASK_HISTORY()` |
| 경량 스케줄링 | Scheduled Notebook | Snowsight UI Schedule 버튼 |
| CI/CD 배포 | GitHub Actions | `snow` CLI + `deploy_dag.py` |

### 다음 단계

- **Module 7**: ML Observability (모델 모니터링, Drift Detection)
- Task 알림 설정 (Slack, Email via Notification Integration)
- Compute Pool GPU 인스턴스를 활용한 딥러닝 학습

In [ ]:
# 리소스 정리 (선택 사항)
# 데모 완료 후 Task Graph를 일시 중단하려면 아래 코드를 실행하세요

def suspend_dag():
    """DAG 스케줄 일시 중단 (Task 정의는 유지, 자동 실행만 중단)"""
    session.sql("ALTER TASK DEMO.ML_DEMO.ML_TRAINING_DAG SUSPEND").collect()
    print("ML_TRAINING_DAG 스케줄 일시 중단 완료")

def resume_dag():
    """DAG 스케줄 재개"""
    session.sql("ALTER TASK DEMO.ML_DEMO.ML_TRAINING_DAG RESUME").collect()
    print("ML_TRAINING_DAG 스케줄 재개 완료")

# 데모 후 자동 실행 중단
# suspend_dag()

print("Module 6 완료!")
print("Task Graph가 배포되어 있습니다. 자동 실행을 중단하려면 suspend_dag()를 호출하세요.")